# 🌸 Data Visualization using Seaborn — Iris Dataset
---
**Author:** Data Science Lab  
**Dataset:** Iris Flower Dataset (150 samples · 3 species · 4 features)  
**Library:** Seaborn (built on Matplotlib)

> **Seaborn** is a Python data-visualization library that provides a high-level interface for drawing attractive and informative statistical graphics. It builds on top of Matplotlib and integrates closely with Pandas DataFrames.

---


* ##  Importing Libraries

We import visualization libraries (**Seaborn**, **Matplotlib**), data-processing libraries (**Pandas**, **NumPy**), and Scikit-learn modules for model building, evaluation, and preprocessing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "legend.fontsize": 10})

---
* ##  Loading & Preparing the Dataset

We load the Iris CSV, drop the index column, rename features for cleanliness, and strip the `Iris-` prefix from species labels. We then encode the target column into integers (required by most sklearn models).

 Species | Description |
|---|---|
| *Iris-setosa* | Easily separable — small petals |
| *Iris-versicolor* | Overlaps slightly with virginica |
| *Iris-virginica* | Largest petals among the three |

Each sample records **four measurements** (in centimetres):
- Sepal Length & Sepal Width
- Petal Length & Petal Width

In [ ]:
df = pd.read_csv('data\Iris.csv')
df.drop(columns=['Id'], inplace=True)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df['species'] = df['species'].str.replace('Iris-', '', regex=False)


le = LabelEncoder()
df['species_enc'] = le.fit_transform(df['species'])   
class_names = le.classes_

print(f"Shape          : {df.shape}")
print(f"Species labels : {class_names.tolist()} → {[0,1,2]}")
df.head(8)

---
* ## Exploratory Data Analysis (EDA)

Before building models, it is essential to understand the data — its distributions, feature relationships, and class balance.

In [ ]:
print("── Data Types ──────────────────────────────")
print(df.dtypes)
print()

print("── Class Distribution ──────────────────────")
print(df['species'].value_counts())

The dataset is **perfectly balanced** — exactly 50 samples per species. All four measurement columns are continuous `float64` values, while `species` is a categorical string column.

In [ ]:
print("── Descriptive Statistics ──────────────────────────")
df.describe().round(2)

**Key observations from the statistics:**
- Petal features have far higher variance (std ≈ 1.76 for petal_length) than sepal features.
- Petal width ranges from 0.1 cm (setosa) to 2.5 cm (virginica) — a huge spread.
- Sepal width is the most tightly distributed feature overall.

### Per-Species Summary Table

A grouped aggregation reveals how much each species differs across all four measurements.

In [ ]:
summary = (df.groupby('species')[['sepal_length','sepal_width','petal_length','petal_width']]
             .agg(['mean','std','min','max']).round(2))
summary.columns = ['_'.join(c) for c in summary.columns]
summary.T

**Interpretation:**
- **Setosa** has the smallest petal dimensions but relatively wide sepals.
- **Virginica** has the largest sepal and petal measurements overall.
- **Versicolor** is the intermediate species across all features.

This separation is crucial for building classification models on this data.


### 1. Line Plots

A **line plot** connects data points along an ordered axis. It is useful for spotting trends or relationships between two continuous variables.

#### a. Sepal Length vs. Sepal Width — Per Species

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
species_list = df['species'].unique()
colors = sns.color_palette("Set2", 3)

for ax, sp, col in zip(axes, species_list, colors):
    subset = df[df['species'] == sp].sort_values('sepal_length')
    sns.lineplot(x='sepal_length', y='sepal_width', data=subset,
                 ci=None, color=col, linewidth=2.5, ax=ax)
    ax.set_title(f"Species: {sp.capitalize()}", fontweight='bold')
    ax.set_xlabel("Sepal Length (cm)")
    ax.set_ylabel("Sepal Width (cm)" if ax == axes[0] else "")

plt.suptitle("Sepal Length vs. Sepal Width — Per Species", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

When separated by species, each group shows a **positive correlation** — longer sepals correspond to wider sepals within each species. This is a classic example of **Simpson's Paradox**.

### 2. Scatter Plots

A **scatter plot** shows the relationship between two continuous variables. Each point represents one sample.

#### a. Sepal Length vs. Sepal Width

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(x='sepal_length', y='sepal_width', data=df,
                hue='species', style='species', s=90, ax=ax)
ax.set_title("Scatter Plot: Sepal Length vs. Sepal Width", fontweight='bold')
ax.set_xlabel("Sepal Length (cm)")
ax.set_ylabel("Sepal Width (cm)")
ax.legend(title='Species')
plt.tight_layout()
plt.show()

**Setosa** (top-left cluster) is clearly separated from the other two species. Versicolor and virginica show some overlap — they are harder to separate using sepal dimensions alone.

#### b. Petal Length vs. Petal Width

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(x='petal_length', y='petal_width', data=df,
                hue='species', style='species', s=90, ax=ax)
ax.set_title("Scatter Plot: Petal Length vs. Petal Width", fontweight='bold')
ax.set_xlabel("Petal Length (cm)")
ax.set_ylabel("Petal Width (cm)")
ax.legend(title='Species')
plt.tight_layout()
plt.show()

This plot is the most discriminative scatter of the dataset:
- **Setosa**: bottom-left, clearly isolated
- **Versicolor**: middle cluster
- **Virginica**: top-right, with only slight overlap with versicolor

Petal dimensions are the **best two features** for visual and algorithmic classification.

### 3. Bar Plots

A **bar plot** aggregates a numerical variable by category (default: mean) and displays it as bars with error-bar confidence intervals.

#### a. Mean Sepal Length per Species

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x='species', y='sepal_length', data=df,
            palette='Set2', capsize=0.1, ax=ax)
ax.set_title("Mean Sepal Length per Species", fontweight='bold')
ax.set_xlabel("Species")
ax.set_ylabel("Mean Sepal Length (cm)")
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.05,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

The error bars (confidence intervals) show variability. Virginica has the longest sepal on average (≈6.59 cm), while setosa has the shortest (≈5.01 cm). The bars annotated with mean values help quick comparison.

#### b. All Feature Means per Species (Grouped Bar)


In [ ]:
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
means = df.groupby('species')[features].mean().reset_index()
means_melted = means.melt(id_vars='species', var_name='Feature', value_name='Mean (cm)')

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x='Feature', y='Mean (cm)', hue='species', data=means_melted,
            palette='Set2', capsize=0.08, ax=ax)
ax.set_title("Mean Feature Values per Species — All Features", fontweight='bold')
ax.set_xlabel("Feature")
ax.set_ylabel("Mean Value (cm)")
ax.legend(title='Species')
plt.tight_layout()
plt.show()

Across all four features, **virginica consistently ranks highest**, and **setosa ranks lowest** for petal features while showing relatively wide sepals.

### 4. Box Plots

A **box plot** (box-and-whisker plot) compactly displays the distribution of a numerical variable:
- **Box** = Interquartile Range (Q1 to Q3)
- **Line inside box** = Median (Q2)
- **Whiskers** = 1.5 × IQR beyond Q1 and Q3
- **Dots** = Outliers

#### a. Sepal Width — Single Feature (No Species Split)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(x='sepal_width', data=df, color='skyblue', ax=ax)
ax.set_title("Box Plot: Sepal Width (Overall Distribution)", fontweight='bold')
ax.set_xlabel("Sepal Width (cm)")
plt.tight_layout()
plt.show()

The overall sepal width is centred around 3 cm. A few outliers are visible on both ends, hinting at unusual specimens.

#### b. Individual Feature Box Plots — Four Panels
